In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
SILVER_RELEASE_TABLE = f"{CATALOG}.{SCHEMA}.silver_release_events"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.oss_ecosystem.silver_release_events (
    event_id STRING,
    event_ts TIMESTAMP,
    actor_id BIGINT,
    repo_id BIGINT,
    action STRING,
    release_id BIGINT,
    tag_name STRING,
    is_draft BOOLEAN,
    is_prerelease BOOLEAN,
    source_file STRING,
    source_hour TIMESTAMP,
    ingested_at TIMESTAMP
)
USING DELTA
""")

In [0]:
release_payload_schema = StructType([
    StructField("action", StringType(), True),

    StructField(
        "release",
        StructType([
            StructField("id", LongType(), True),
            StructField("tag_name", StringType(), True),
            StructField("draft", BooleanType(), True),
            StructField("prerelease", BooleanType(), True)
        ]),
        True
    )
])

In [0]:
release_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "ReleaseEvent")
)

print(
    "ReleaseEvent rows in Bronze:",
    release_bronze_df.count()
)

In [0]:
existing_release_ids = (
    spark.table(SILVER_RELEASE_TABLE)
    .select("event_id")
)

new_release_bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "ReleaseEvent")
    .join(
        existing_release_ids,
        on="event_id",
        how="left_anti"
    )
)

print(
    "New ReleaseEvents to transform:",
    new_release_bronze_df.count()
)

In [0]:
silver_release_df = (
    new_release_bronze_df
    .withColumn(
        "payload_parsed",
        F.from_json(
            F.col("payload_json"),
            release_payload_schema
        )
    )
    .select(
        "event_id",

        F.to_timestamp("created_at")
            .alias("event_ts"),

        F.get_json_object("actor_json", "$.id")
            .cast("long")
            .alias("actor_id"),

        F.get_json_object("repo_json", "$.id")
            .cast("long")
            .alias("repo_id"),

        F.col("payload_parsed.action")
            .alias("action"),

        F.col("payload_parsed.release.id")
            .alias("release_id"),

        F.col("payload_parsed.release.tag_name")
            .alias("tag_name"),

        F.col("payload_parsed.release.draft")
            .alias("is_draft"),

        F.col("payload_parsed.release.prerelease")
            .alias("is_prerelease"),

        "source_file",
        "source_hour",
        "ingested_at"
    )
)

In [0]:
print(
    "Silver Release rows:",
    silver_release_df.count()
)

silver_release_df.select(
    F.sum(F.col("event_id").isNull().cast("int"))
        .alias("null_event_id"),

    F.sum(F.col("actor_id").isNull().cast("int"))
        .alias("null_actor_id"),

    F.sum(F.col("repo_id").isNull().cast("int"))
        .alias("null_repo_id"),

    F.sum(F.col("action").isNull().cast("int"))
        .alias("null_action"),

    F.sum(F.col("release_id").isNull().cast("int"))
        .alias("null_release_id"),

    F.sum(F.col("tag_name").isNull().cast("int"))
        .alias("null_tag_name"),

    F.sum(F.col("is_draft").isNull().cast("int"))
        .alias("null_is_draft"),

    F.sum(F.col("is_prerelease").isNull().cast("int"))
        .alias("null_is_prerelease")
).show()

In [0]:
duplicate_count = (
    silver_release_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate ReleaseEvent IDs:",
    duplicate_count
)

In [0]:
if not spark.catalog.tableExists(SILVER_RELEASE_TABLE):
    (
        silver_release_df.limit(0)
        .write
        .format("delta")
        .saveAsTable(SILVER_RELEASE_TABLE)
    )

silver_release_delta = DeltaTable.forName(
    spark,
    SILVER_RELEASE_TABLE
)

(
    silver_release_delta.alias("t")
    .merge(
        silver_release_df.alias("s"),
        "t.event_id = s.event_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver Release MERGE completed.")

In [0]:
print(
    "Silver Release table rows:",
    spark.table(SILVER_RELEASE_TABLE).count()
)

print(
    "Duplicate ReleaseEvent IDs:",
    spark.table(SILVER_RELEASE_TABLE)
        .groupBy("event_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)